# Feature UMAP Dashboard — "Old Method" Baseline

Loads precomputed geometric + texture features from `cell_table.csv`,
filters edge-cropped cells, runs UMAP, and saves an interactive HTML dashboard.

Features used (34 total, from `ComputeFeatures.py`):
- `gFeat_cell_*` — 12 regionprops metrics on pCellmask (area, circularity, eccentricity …)
- `gFeat_nuc_*`  — 12 regionprops metrics on pNucmask
- `tFeat_mem_*`  — 5 texture metrics on membrane channel (entropy, GLCM)
- `tFeat_nuc_*`  — 5 texture metrics on nuclei channel

Cells with any NaN feature are dropped (not just edge-filtered).
Features are z-scored before UMAP.

In [4]:
import sys, io, json, base64
from pathlib import Path

import numpy as np
import pandas as pd
import umap
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go

try:
    from tqdm.notebook import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

In [5]:
# ── config ────────────────────────────────────────────────────────────────────
TABLE          = "/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/cell_table.csv"
OUT_DIR        = Path("outputs/feature_umap")
EDGE_THRESHOLD = 5       # must match training (edge_run_px >= N → drop)
# UMAP
N_NEIGHBORS = 15
MIN_DIST    = 0.1
SEED        = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output : {OUT_DIR.resolve()}")

Output : /mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/Joaquin'scripts/outputs/feature_umap


In [6]:
# ── load & filter ─────────────────────────────────────────────────────────────
df = pd.read_csv(TABLE)
print(f"Loaded {len(df)} cells from {TABLE}")

# edge filter
if "edge_run_px" not in df.columns:
    raise RuntimeError("edge_run_px column missing — run add_edge_flag.py first")
df = df[df["edge_run_px"] < EDGE_THRESHOLD].reset_index(drop=True)
print(f"After edge filter (< {EDGE_THRESHOLD} px): {len(df)} cells")

# select feature columns
feat_cols = [c for c in df.columns if c.startswith(("gFeat_", "tFeat_"))]
if not feat_cols:
    raise RuntimeError("No gFeat_/tFeat_ columns found — run ComputeFeatures.py first")
print(f"Feature columns ({len(feat_cols)}): {feat_cols[:6]} …")

# drop orientation (circular, not meaningful in Euclidean distance)
feat_cols = [c for c in feat_cols if "orientation" not in c]

# drop cells with any NaN feature
n_before = len(df)
df = df.dropna(subset=feat_cols).reset_index(drop=True)
print(f"After NaN drop: {len(df)} cells ({n_before - len(df)} removed)")
print(f"Features used: {len(feat_cols)}")

Loaded 16678 cells from /mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/cell_table.csv
After edge filter (< 5 px): 15210 cells
Feature columns (34): ['gFeat_cell_n_pixels', 'gFeat_cell_area', 'gFeat_cell_perimeter', 'gFeat_cell_circularity', 'gFeat_cell_solidity', 'gFeat_cell_convex_area'] …
After NaN drop: 14985 cells (225 removed)
Features used: 32


In [7]:
# ── scale + UMAP ──────────────────────────────────────────────────────────────
X = df[feat_cols].values.astype(np.float32)
X = StandardScaler().fit_transform(X)

reducer = umap.UMAP(
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    n_components=2,
    random_state=SEED,
    verbose=True,
)
xy = reducer.fit_transform(X)
print(f"UMAP done: {xy.shape}")

/home/S-JS/conda/envs/darth-vaeder/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(n_jobs=1, random_state=42, verbose=True)
Wed Jun 17 16:39:15 2026 Construct fuzzy simplicial set
Wed Jun 17 16:39:15 2026 Finding Nearest Neighbors
Wed Jun 17 16:39:15 2026 Building RP forest with 11 trees
Wed Jun 17 16:39:21 2026 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	Stopping threshold met -- exiting after 3 iterations
Wed Jun 17 16:39:35 2026 Finished Nearest Neighbor Search
Wed Jun 17 16:39:40 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Wed Jun 17 16:39:57 2026 Finished embedding
UMAP done: (14985, 2)


In [8]:
# ── interactive Plotly scatter (notebook) ─────────────────────────────────────
PAL = [
    "#4e9de0", "#ff8c42", "#e040fb", "#d62728", "#2ca02c",
    "#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf",
]

conditions = list(df["condition"].unique())
cond_to_idx = {c: df.index[df["condition"] == c].tolist() for c in conditions}

fig = go.Figure()
for i, cond in enumerate(conditions):
    idx = cond_to_idx[cond]
    fig.add_trace(go.Scatter(
        x=xy[idx, 0], y=xy[idx, 1],
        mode="markers",
        name=cond,
        marker=dict(size=4, opacity=0.75, color=PAL[i % len(PAL)]),
        customdata=np.array(idx),
        hovertemplate=f"<b>{cond}</b><br>idx=%{{customdata}}<extra></extra>",
    ))

fig.update_layout(
    title=f"Feature UMAP  ({len(feat_cols)} features, {len(df)} cells)",
    height=700,
    dragmode="pan",
    margin=dict(l=50, r=20, t=60, b=50),
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(title="UMAP 1", gridcolor="#eee", zeroline=False),
    yaxis=dict(title="UMAP 2", gridcolor="#eee", zeroline=False),
    legend=dict(bgcolor="white", bordercolor="#ddd", borderwidth=1),
)
fig.show()

In [10]:
# ── build + save standalone HTML dashboard ────────────────────────────────────
# Click a point to see its condition, replicate, and top feature values.

# build per-cell metadata for the panel
classes = [
    {
        "name":    cond,
        "x":       xy[cond_to_idx[cond], 0].tolist(),
        "y":       xy[cond_to_idx[cond], 1].tolist(),
        "indices": cond_to_idx[cond],
    }
    for cond in conditions
]

# store a small subset of features per cell for the hover panel
PANEL_FEATS = [
    "gFeat_cell_area", "gFeat_cell_circularity", "gFeat_cell_eccentricity",
    "gFeat_nuc_area",  "gFeat_nuc_circularity",
    "tFeat_mem_entropy", "tFeat_mem_glcm_contrast",
    "tFeat_nuc_entropy", "tFeat_nuc_glcm_contrast",
]
PANEL_FEATS = [f for f in PANEL_FEATS if f in df.columns]

items = [
    {
        "condition": str(row["condition"]),
        "replicate": str(row["replicate"]),
        "filename":  f"{row['replicate']}|{row['condition']}|{row['image_name']}|cell_{row['local_cell_index']}",
        "feats":     {f: round(float(row[f]), 4) for f in PANEL_FEATS},
    }
    for _, row in df.iterrows()
]

data_dict = {
    "n_feats": len(feat_cols),
    "n_cells": len(items),
    "classes": classes,
    "items":   items,
    "panel_feats": PANEL_FEATS,
}

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <title>Feature UMAP — Old Method Baseline</title>
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
  <style>
    *, *::before, *::after { box-sizing: border-box }
    body  { font-family: Arial, sans-serif; margin: 0; padding: 16px;
            background: #f0f0f0; color: #222 }
    h2    { margin: 0 0 4px }
    .sub  { color: #999; font-size: 13px; margin: 0 0 14px }
    #wrap      { display: flex; gap: 16px; align-items: flex-start }
    #plot-col  { flex: 3 }
    #panel-col { flex: 2; min-width: 340px; max-height: 760px;
                 overflow-y: auto; background: #fff; border-radius: 8px;
                 padding: 14px; box-shadow: 0 2px 8px #bbb }
    #panel-col h4 { margin: 0 0 10px; font-size: 15px }
    .hint { color: #ccc; font-style: italic }
    .ibox b   { font-size: 14px }
    .ibox .meta { font-size: 11px; color: #999; word-break: break-all; margin: 2px 0 8px }
    table { border-collapse: collapse; width: 100%; font-size: 12px; margin-top: 6px }
    th { text-align: left; color: #888; font-weight: normal;
         border-bottom: 1px solid #eee; padding: 3px 6px }
    td { padding: 3px 6px; border-bottom: 1px solid #f4f4f4 }
    td:last-child { text-align: right; font-family: monospace }
    .ghdr { font-size: 13px; font-weight: bold; margin-bottom: 8px }
    .grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 6px }
    .tile { background: #f8f8f8; border-radius: 3px; padding: 6px;
            font-size: 11px; color: #555 }
  </style>
</head>
<body>
<h2>Feature UMAP — Handcrafted Features Baseline</h2>
<p class="sub">
  Click a point to view its feature values &nbsp;&middot;&nbsp;
  Box/lasso-select for a summary &nbsp;&middot;&nbsp;
  Scroll to zoom &nbsp;&middot;&nbsp; Drag to pan<br>
  <small>__N_FEATS__ features (geometry + texture) &nbsp;|&nbsp; __N_CELLS__ cells</small>
</p>
<div id="wrap">
  <div id="plot-col"><div id="umap-plot"></div></div>
  <div id="panel-col">
    <h4>Cell Feature Viewer</h4>
    <div id="panel-content">
      <p class="hint">&#x1F446; Click a point to inspect its features.</p>
    </div>
  </div>
</div>
<script>
const DATA = __DATA_JSON__;
const PAL = [
  "#4e9de0","#ff8c42","#e040fb","#d62728","#2ca02c",
  "#9467bd","#8c564b","#e377c2","#bcbd22","#17becf"
];
const traces = DATA.classes.map((cls, i) => ({
  x: cls.x, y: cls.y,
  mode: "markers", type: "scatter", name: cls.name,
  customdata: cls.indices,
  marker: { size: 4, opacity: 0.75, color: PAL[i % PAL.length] },
  hovertemplate: "<b>" + cls.name + "</b><br>idx=%{customdata}<extra></extra>",
}));
Plotly.newPlot("umap-plot", traces, {
  title: { text: "Feature UMAP  (" + DATA.n_feats + " features)", font: { size: 16 } },
  height: 720, clickmode: "event+select", dragmode: "pan",
  margin: { l: 50, r: 20, t: 50, b: 50 },
  plot_bgcolor: "#fff", paper_bgcolor: "#fff",
  xaxis: { title: "UMAP 1", gridcolor: "#eee" },
  yaxis: { title: "UMAP 2", gridcolor: "#eee" },
  legend: { bgcolor: "#fff", bordercolor: "#ddd", borderwidth: 1 },
}, { scrollZoom: true, responsive: true });
const panel = document.getElementById("panel-content");
function singleView(globalIdx) {
  const it = DATA.items[globalIdx];
  let rows = DATA.panel_feats.map(f =>
    `<tr><td>${f.replace('gFeat_cell_','cell·').replace('gFeat_nuc_','nuc·').replace('tFeat_mem_','mem·').replace('tFeat_nuc_','nuc·')}</td><td>${it.feats[f] ?? '—'}</td></tr>`
  ).join('');
  panel.innerHTML =
    `<div class="ibox">
       <b>${it.condition} — ${it.replicate}</b>
       <div class="meta">${it.filename}</div>
     </div>
     <table><tr><th>feature</th><th>value</th></tr>${rows}</table>`;
}
function gridView(pts) {
  const MAX = 8, n = pts.length;
  const counts = {};
  pts.forEach(p => { const c = DATA.items[p.customdata].condition; counts[c] = (counts[c]||0)+1; });
  let h = `<div class="ghdr">${n} point${n!==1?'s':''} selected</div>`
         + `<div class="grid">`
         + Object.entries(counts).map(([c,n]) =>
             `<div class="tile"><b>${c}</b><br>${n} cell${n!==1?'s':''}</div>`).join('')
         + `</div>`;
  panel.innerHTML = h;
}
const el = document.getElementById("umap-plot");
el.on("plotly_click",    ev => singleView(ev.points[0].customdata));
el.on("plotly_selected", ev => { if (ev && ev.points.length) gridView(ev.points); });
</script>
</body>
</html>"""

data_json = json.dumps(data_dict, separators=(",", ":"))
html_out  = (HTML_TEMPLATE
             .replace("__DATA_JSON__", data_json)
             .replace("__N_FEATS__", str(len(feat_cols)))
             .replace("__N_CELLS__", str(len(items))))

out_path = OUT_DIR / "feature_umap_dashboard.html"
out_path.write_text(html_out, encoding="utf-8")
size_mb = out_path.stat().st_size / 1e6
print(f"Saved: {out_path}  ({size_mb:.1f} MB)")

Saved: outputs/feature_umap/feature_umap_dashboard.html  (6.1 MB)
